In [2]:
"""
Descarga de datos de generación eléctrica por tipo de producción
Sistema PENINSULAR de España, del 01/10/2020 al 31/12/2020.

Fuente: API REData de Red Eléctrica de España (REE) - acceso público, sin token.
Documentación: https://www.ree.es/es/apidatos
Endpoint: generacion/estructura-generacion
"""

import requests
import pandas as pd
import time

BASE_URL = "https://apidatos.ree.es/es/datos/generacion/estructura-generacion"

# Parámetros de la consulta
params = {
    "start_date": "2020-10-01T00:00",
    "end_date": "2020-12-31T23:59",
    "time_trunc": "day",          # granularidad: day / hour / month
    "geo_trunc": "electric_system",
    "geo_limit": "peninsular",    # sistema peninsular
    "geo_ids": 8741                # 8741 = código geo_id de la España peninsular en REData
}

headers = {
    "Accept": "application/json; application/vnd.api+json",
    "Content-Type": "application/json"
}


def descargar_generacion(params, headers, reintentos=3):
    for intento in range(reintentos):
        resp = requests.get(BASE_URL, params=params, headers=headers, timeout=30)
        if resp.status_code == 200:
            return resp.json()
        print(f"Intento {intento+1}: status {resp.status_code}, reintentando...")
        time.sleep(2)
    resp.raise_for_status()


def json_a_dataframe(data):
    filas = []
    for serie in data.get("included", []):
        tipo = serie["attributes"]["title"]  # p.ej. "Nuclear", "Eólica", "Solar fotovoltaica"...
        for punto in serie["attributes"]["values"]:
            filas.append({
                "fecha": punto["datetime"],
                "tipo_generacion": tipo,
                "valor_MWh": punto["value"],
                "porcentaje": punto.get("percentage")
            })
    df = pd.DataFrame(filas)
    df["fecha"] = pd.to_datetime(df["fecha"], utc=True).dt.tz_localize(None)
    return df.sort_values(["tipo_generacion", "fecha"]).reset_index(drop=True)


if __name__ == "__main__":
    print("Descargando datos de REData (REE)...")
    data = descargar_generacion(params, headers)
    df = json_a_dataframe(data)

    print(f"\nRegistros descargados: {len(df)}")
    print(f"Tipos de generación encontrados: {df['tipo_generacion'].unique().tolist()}")
    print(f"Rango de fechas: {df['fecha'].min()} - {df['fecha'].max()}")

    # Guardar en CSV
    salida = "generacion_peninsula_2020Q4_REData.csv"
    df.to_csv(salida, index=False, encoding="utf-8-sig")
    print(f"\nGuardado en: {salida}")

    # Tabla resumen: total generado por tipo en el periodo
    resumen = (
        df.groupby("tipo_generacion")["valor_MWh"]
        .sum()
        .sort_values(ascending=False)
        .reset_index()
    )
    print("\nResumen total por tipo de generación (MWh):")
    print(resumen.to_string(index=False))

Descargando datos de REData (REE)...

Registros descargados: 1104
Tipos de generación encontrados: ['Carbón', 'Ciclo combinado', 'Cogeneración', 'Eólica', 'Generación total', 'Hidráulica', 'Nuclear', 'Otras renovables', 'Residuos no renovables', 'Residuos renovables', 'Solar fotovoltaica', 'Solar térmica']
Rango de fechas: 2020-09-30 22:00:00 - 2020-12-30 23:00:00

Guardado en: generacion_peninsula_2020Q4_REData.csv

Resumen total por tipo de generación (MWh):
       tipo_generacion    valor_MWh
      Generación total 60943107.589
                Eólica 17207385.311
               Nuclear 14440509.179
       Ciclo combinado  8577247.906
            Hidráulica  7560522.729
          Cogeneración  7095033.404
    Solar fotovoltaica  2796672.512
      Otras renovables  1230425.072
                Carbón   793117.368
         Solar térmica   524548.891
Residuos no renovables   518436.831
   Residuos renovables   199208.386
